# Session 3 — Signal Region and Analysis Strategy

Here we combine the object selections from Session 2 into a **full signal region (SR)** for the bb + MET search. We will also look at how the main backgrounds populate the SR and how control regions are used conceptually.

**Learning objectives:**
- Define and implement the bb + MET signal region for the bbDM search
- Apply full event selection and study the resulting MET distribution
- Compare event yields before and after cuts (cutflow and yield tables)
- Understand background composition and the role of control regions

<div style="display: flex; gap: 0; align-items: flex-start; justify-content: center; margin-top: 0.75rem; flex-wrap: wrap;">
  <div style="flex: 1 1 45%; min-width: 260px; text-align: right;">
    <img src="figures/session3_fig_regions_cartoon.png" alt="Regions cartoon (SR and control regions)" style="width: 100%; max-width: 480px; height: auto; display: block; margin: 0 auto;"/>
  </div>
  <div style="flex: 1 1 45%; min-width: 260px; text-align: left;">
    <img src="figures/session3_fig_sr_cartoon.png" alt="Signal region cartoon" style="width: 100%; max-width: 480px; height: auto; display: block; margin: 0 auto;"/>
  </div>
</div>


---
## 1. Signal Region Definition

We define the **signal region (SR)** as:

- **Trigger:** pass at least one of the analysis HLT paths (MET / muon )

- **2 ≤ Njets ≤ 3**, with the two leading jets b-tagged (medium WP)

- **0 isolated leptons** (lepton veto)

- **Δφ(jet, MET) > 0.5**

- **$\Delta p_\mathrm{T}^\mathrm{miss}(\mathrm{PF} - \mathrm{calo}) < 0.5$ GeV** (after Δφ; before the MET cut; skipped if CaloMET is absent from the file)

- **MET > 250 GeV**

This targets the signature $pp → b\bar{b}+\chi\bar{\chi}$ while suppressing many backgrounds.

---
## 2. Backgrounds in the Signal Region

| Background | Why it enters | How we suppress |
|------------|----------------|------------------|
| **$t\bar{t}$** | Has b-jets; semileptonic has MET from ν | Lepton veto removes most; MET cut reduces rest |
| **Z $(→\nu\bar{\nu})$ +jets** | Invisible $\nu\bar{\nu}→$ large MET | Dominant at high MET; need MC/data-driven estimate |
| **W + jets** | W $→$ $\ell$ $\nu$ gives MET | Lepton veto removes W $→$ $\ell$ $\nu$ |


In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt

from config.datasets_2017 import get_short_datasets_meta, get_trigger_list
from processor.bbdm_processor import get_nanoevents, select_good_jets, count_bjets, n_tight_leptons, n_leptons, RECOIL_MIN, cos_theta_star, recoil_pt, get_recoil, met_pf_calo_mask

# Normalization
LUMI_PB = 41.5 * 1000

# Notebook-level controls for short YAML loading
# - NFILES_PER_DATASET: use first N files from each dataset in short YAML (None = use all listed)
# - MAX_EVENTS: keep first N events after concatenating files (<=0 = use all events)
NFILES_PER_DATASET = 3
MAX_EVENTS = 10_000
datasets = get_short_datasets_meta()

events_by_dataset = {}
labels = {}
xsecs = {}
is_data = {}
norm_factors = {}

print(f"[setup] loading datasets from short YAML: {len(datasets)} entries")
for name, meta in datasets.items():
    labels[name] = meta.get("label", name)
    xsecs[name] = meta.get("xsec")
    is_data[name] = bool(meta.get("isData", False))
    path_list = meta.get("files", [])
    if NFILES_PER_DATASET is not None and NFILES_PER_DATASET > 0:
        path_list = path_list[:NFILES_PER_DATASET]

    print(f"[dataset] {name}: requested_files={len(path_list)}")
    if path_list:
        ev_chunks = []
        for i, p in enumerate(path_list, start=1):
            print(f"  -> loading file {i}/{len(path_list)}: {p}")
            try:
                ev_chunks.append(get_nanoevents(p))
                print("     loaded")
            except Exception as e:
                print(f"     failed: {e}")
                continue
        if not ev_chunks:
            print(f"  !! no files loaded for {name}, skipping")
            continue
        ev = ak.concatenate(ev_chunks, axis=0) if len(ev_chunks) > 1 else ev_chunks[0]
        if MAX_EVENTS is not None and MAX_EVENTS > 0:
            ev = ev[:MAX_EVENTS]
        events_by_dataset[name] = ev
        print(f"  == events loaded for {name}: {len(ev)}")

        if not is_data[name]:
            xsec = xsecs.get(name)
            if xsec is None:
                norm_factors[name] = None
            else:
                norm_factors[name] = (float(xsec) * LUMI_PB) / float(len(ev))

print(f"[setup] done. datasets loaded: {len(events_by_dataset)}")

# Recoil helper is provided by processor as `get_recoil(events)`.

# MC-only for SR: background names (exclude data and signal)
bkg_names = [k for k in events_by_dataset if (not is_data.get(k, False)) and ("bbDM" not in k)]

# Matplotlib legend labels (LaTeX)
latex_labels = {
    "DYJets": r"$Z(\ell\ell)+$jets",
    "ZJets": r"$Z(\nu\bar{\nu})+$jets",
    "WJets": r"$W(\ell\nu)+$jets",
    "DIBOSON": r"WW",
    "STop": r"Single $t$",
    "Top": r"$t\bar{t}$",
    "SMH": r"SMH",
    "bbDM_2HDMa_LO_5f": r"bbDM (2HDM+a)",
    "MET_Run2017B": r"Data MET",
    "SingleElectron_Run2017": r"Data SingleElectron",
}

PROCESS_COLORS = {
    "DYJets": "#3BAF2A",
    "ZJets": "#2A64AD",
    "WJets": "#8B4FC9",
    "DIBOSON": "#1F3FB3",
    "STop": "#F39C34",
    "Top": "#D97A00",
    "SMH": "#C62828",
    "QCD": "#6E6E6E",
}
DEFAULT_STACK_COLOR = "#9A9A9A"

def legend_stack():
    plt.legend(loc="upper right", ncol=2, fontsize=9, frameon=False, columnspacing=1.0, handlelength=1.6)


# Sanity printout: xsec, N, normalization factor
for name in bkg_names:
    print(name, "xsec=", xsecs.get(name), "N=", len(events_by_dataset[name]), "norm=", norm_factors.get(name))

---
## 3. Full Event Selection

Combine all steps:
1. **Trigger** (OR of analysis HLT paths)

2. Good jets (pT, η, jetId); 2 ≤ Njets ≤ 3, leading two b-tagged

3. Count tight leptons (veto: require 0)

4. Δφ(jet, MET) > 0.5

5. $\Delta p_\mathrm{T}^\mathrm{miss}(\mathrm{PF} - \mathrm{calo}) < 0.5$

6. MET > 250 GeV

Implement this and count events passing each step (cutflow).

In [ ]:
# Run cutflow for each dataset (events_by_dataset and labels from setup cell)
def run_cutflow(events):
    # Trigger OR (first cut)
    trigger_list = get_trigger_list()
    hlt_fields = set(events.HLT.fields) if hasattr(events, "HLT") and hasattr(events.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(events.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | events.HLT[tname]
    n0 = int(ak.sum(trigger_mask))
    events = events[trigger_mask]

    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    nlep = n_leptons(events)
    # Recoil is precomputed for all events before SR/CR masks.
    recoil = get_recoil(events)
    met_phi = events.MET.phi
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5

    jets_2to3 = (njets >= 2) & (njets <= 3)
    sr_jets = jets_2to3 & (nbjets == 2)

    n1 = ak.sum(jets_2to3)
    n1b = ak.sum(sr_jets)
    n2 = ak.sum(sr_jets & (nlep == 0))
    n3 = ak.sum(sr_jets & (nlep == 0) & dphi_cut)
    met_pf_calo_ok = met_pf_calo_mask(events)
    n4 = ak.sum(sr_jets & (nlep == 0) & dphi_cut & met_pf_calo_ok)
    n5 = ak.sum(sr_jets & (nlep == 0) & dphi_cut & met_pf_calo_ok & (recoil > RECOIL_MIN))
    recoil_key = f"Recoil>{int(RECOIL_MIN)}"
    return {
        "trigger": n0,
        "njet_2to3": int(n1),
        "nbjet_eq2": int(n1b),
        "0lep": int(n2),
        "Δφ>0.5": int(n3),
        "|ΔMET(PF-calo)|<0.5": int(n4),
        recoil_key: int(n5),
    }

cutflow_by_dataset = {name: run_cutflow(ev) for name, ev in events_by_dataset.items()}
for name in cutflow_by_dataset:
    print(labels.get(name, name), cutflow_by_dataset[name])

signal_names = [k for k in cutflow_by_dataset if "bbDM" in k or k.startswith("signal_")]
if signal_names:
    print("\nSignal cutflow:")
    for n in signal_names:
        print("  ", labels.get(n, n), cutflow_by_dataset[n])


### Exercise 3.1 — Cutflow
Implement the cutflow above and fill a table: **Cut** | **Yield**. Use your NanoAOD sample (one process is enough for the exercise).

In [ ]:
# Your code: cutflow table

---
## 4. Recoil Distribution in the Signal Region

**Note:** Signal-region plots are **MC-only**; no data is plotted in the signal region.

Plot **recoil** for events that pass the full signal region (same cuts as the cutflow: after Δφ and PF–calo consistency, **recoil > 250 GeV**, ≥2 b-jets, 0 leptons). Use bins `[250, 300, 400, 550, 1000]`. Use `events_by_dataset` and `bkg_names` to stack background MC.

In [ ]:
# Recoil in SR: stack backgrounds (scaled to 41.5/fb) + overlay selected signal
bins_met = np.array([250, 300, 400, 550, 1000], dtype=float)
met_arrays, w_arrays, leg_labels, met_keys = [], [], [], []

trigger_list = get_trigger_list()
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue

    ev = events_by_dataset[name]
    hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev.HLT[tname]
    ev = ev[trigger_mask]
    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    nlep = n_leptons(ev)
    met_phi = ev.MET.phi
    recoil_evt = get_recoil(ev)

    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5


    sr_mask = (
        (njets >= 2)
        & (njets <= 3)
        & (nbjets == 2)
        & (nlep == 0)
        & dphi_cut
        & met_pf_calo_mask(ev)
        & (recoil_evt > RECOIL_MIN)
    )
    recoil = recoil_evt[sr_mask]
    flat = ak.to_numpy(ak.ravel(recoil))
    if len(flat) == 0:
        continue

    met_arrays.append(flat)
    w_arrays.append(np.full_like(flat, norm, dtype=float))
    leg_labels.append(latex_labels.get(name, labels.get(name, name)))
    met_keys.append(name)

plt.figure()
if met_arrays:
    plt.hist(met_arrays, bins=bins_met, weights=w_arrays, label=leg_labels, stacked=True, histtype="stepfilled", alpha=0.7)

sig_name = "bbDM_2HDMa_LO_5f"
sig_masspoints = [
    ("Signal mA=600, ma=250", "MH3_600_MH4_250_Mchi_1", 0.12550),
    ("Signal mA=600, ma=500", "MH3_600_MH4_500_Mchi_1", 0.0257900),
]
if sig_name in events_by_dataset:
    ev_all = events_by_dataset[sig_name]
    gm_fields = set(ev_all.GenModel.fields) if hasattr(ev_all, "GenModel") and hasattr(ev_all.GenModel, "fields") else set()

    for sig_label, gm_field, sig_xsec_pb in sig_masspoints:
        if gm_field not in gm_fields:
            continue

        mp_all_mask = ak.fill_none(ev_all.GenModel[gm_field] != 0, False)
        n_gen = int(ak.sum(mp_all_mask))
        if n_gen <= 0:
            continue

        ev = ev_all[mp_all_mask]
        hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
        trigger_mask = ak.zeros_like(ev.event, dtype=bool)
        for tname in trigger_list:
            if tname in hlt_fields:
                trigger_mask = trigger_mask | ev.HLT[tname]
        if ak.sum(trigger_mask) == 0:
            trigger_mask = ak.ones_like(ev.event, dtype=bool)
        ev = ev[trigger_mask]

        good_jets = select_good_jets(ev)
        njets = ak.num(good_jets)
        nbjets = count_bjets(good_jets)
        nlep = n_leptons(ev)
        met_phi = ev.MET.phi
        recoil_evt = get_recoil(ev)
        dphi = np.abs(good_jets.phi - met_phi)
        dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
        min_dphi = ak.min(dphi, axis=1)
        dphi_cut = min_dphi > 0.5
        sr_mask = (
            (njets >= 2)
            & (njets <= 3)
            & (nbjets == 2)
            & (nlep == 0)
            & dphi_cut
            & met_pf_calo_mask(ev)
            & (recoil_evt > RECOIL_MIN)
        )

        vals = ak.to_numpy(ak.ravel(recoil_evt[sr_mask]))
        if len(vals) == 0:
            continue

        sig_norm = (float(sig_xsec_pb) * LUMI_PB) / float(n_gen)
        sig_w = np.full_like(vals, sig_norm, dtype=float)
        plt.hist(vals, bins=bins_met, weights=sig_w, histtype="step", linewidth=2.0, label=sig_label)
ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("Recoil [GeV]")
plt.ylabel("Events (scaled)")
plt.title("Recoil in signal region (MC + selected signal overlays)")
legend_stack()
plt.show()

### Exercise 3.2 — MET after selection
Produce the MET distribution in the signal region. Add axis labels and a title. If you have multiple samples (e.g. tt̄, Z→νν), you can plot them stacked or overlaid.

In [ ]:
# Your code: MET in SR

---
## 5. cos θ* distribution in the signal region

The **cos θ*** observable is defined from the pseudorapidity difference of the two leading jets: **cos θ* = |tanh(Δη/2)|**, with Δη = η_jet0 − η_jet1. It is the main angular observable for the bb+MET search and is used in the fit and limit in Session 4.

Below we plot the cos θ* distribution for events passing the full signal region (same selection as the MET plot). Only MC backgrounds are stacked; the same luminosity scaling (41.5/fb) is applied.

In [ ]:
# cos(theta*) in SR (main observable): stack backgrounds + overlay two selected signal mass points
trigger_list = get_trigger_list()
bins_cts = np.linspace(0, 1, 5)
cts_arrays, cts_w, cts_labels, cts_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    ev = events_by_dataset[name]
    hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev.HLT[tname]
    ev = ev[trigger_mask]
    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    nlep = n_leptons(ev)
    met_phi = ev.MET.phi
    recoil_evt = get_recoil(ev)
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5
    sr_mask = (
        (njets >= 2)
        & (njets <= 3)
        & (nbjets == 2)
        & (nlep == 0)
        & dphi_cut
        & met_pf_calo_mask(ev)
        & (recoil_evt > RECOIL_MIN)
    )
    good_jets_sr = good_jets[sr_mask]
    has_two = ak.num(good_jets_sr) >= 2
    jets_pad = ak.pad_none(good_jets_sr, 2)
    jet0 = jets_pad[:, 0]
    jet1 = jets_pad[:, 1]
    mask = has_two & ~ak.is_none(jet1)
    if ak.sum(mask) == 0:
        continue
    cts = cos_theta_star(jet0[mask], jet1[mask])
    flat = ak.to_numpy(ak.ravel(cts))
    if len(flat) == 0:
        continue
    cts_arrays.append(flat)
    cts_w.append(np.full_like(flat, norm, dtype=float))
    cts_labels.append(latex_labels.get(name, labels.get(name, name)))
    cts_keys.append(name)

plt.figure()
if cts_arrays:
    plt.hist(cts_arrays, bins=bins_cts, weights=cts_w, label=cts_labels, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in cts_keys], stacked=True, histtype="stepfilled", alpha=0.9)

# Overlay only requested signal mass points from randomized signal file
sig_name = "bbDM_2HDMa_LO_5f"
sig_masspoints = [
    ("Signal mA=600, ma=250", "MH3_600_MH4_250_Mchi_1", 0.12550),
    ("Signal mA=600, ma=500", "MH3_600_MH4_500_Mchi_1", 0.0257900),
]
if sig_name in events_by_dataset:
    ev_all = events_by_dataset[sig_name]
    gm_fields = set(ev_all.GenModel.fields) if hasattr(ev_all, "GenModel") and hasattr(ev_all.GenModel, "fields") else set()

    for sig_label, gm_field, sig_xsec_pb in sig_masspoints:
        if gm_field not in gm_fields:
            continue

        mp_all_mask = ak.fill_none(ev_all.GenModel[gm_field] != 0, False)
        n_gen = int(ak.sum(mp_all_mask))
        if n_gen <= 0:
            continue

        ev = ev_all[mp_all_mask]
        hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
        trigger_mask = ak.zeros_like(ev.event, dtype=bool)
        for tname in trigger_list:
            if tname in hlt_fields:
                trigger_mask = trigger_mask | ev.HLT[tname]
        if ak.sum(trigger_mask) == 0:
            trigger_mask = ak.ones_like(ev.event, dtype=bool)
        ev = ev[trigger_mask]

        good_jets = select_good_jets(ev)
        njets = ak.num(good_jets)
        nbjets = count_bjets(good_jets)
        nlep = n_leptons(ev)
        met_phi = ev.MET.phi
        recoil_evt = get_recoil(ev)
        dphi = np.abs(good_jets.phi - met_phi)
        dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
        min_dphi = ak.min(dphi, axis=1)
        dphi_cut = min_dphi > 0.5
        sr_mask = (
            (njets >= 2)
            & (njets <= 3)
            & (nbjets == 2)
            & (nlep == 0)
            & dphi_cut
            & met_pf_calo_mask(ev)
            & (recoil_evt > RECOIL_MIN)
        )
        gj = good_jets[sr_mask]
        has_two = ak.num(gj) >= 2
        jp = ak.pad_none(gj, 2)
        j0 = jp[:, 0]
        j1 = jp[:, 1]
        valid = has_two & ~ak.is_none(j1)
        if ak.sum(valid) == 0:
            continue

        vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
        if len(vals) == 0:
            continue

        sig_norm = (float(sig_xsec_pb) * LUMI_PB) / float(n_gen)
        sw = np.full_like(vals, sig_norm, dtype=float)
        plt.hist(vals, bins=bins_cts, weights=sw, histtype="step", linewidth=2.0, label=sig_label)
ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("$\\cos(\\theta)*$ (SR)")
plt.ylabel("Events (scaled)")
plt.title("$\\cos(\\theta)*$ in signal region (MC + selected signal overlays)")
legend_stack()
plt.show()

---
## 6. Yields Before and After Cuts

Build a **yield table** with one row per cut and one column per process (if you have multiple samples). Example:

| Cut | $t\bar{t}$ | Z ( $→\nu\bar{\nu}$)+jets | W( $→\ell\nu$)+jets |
|-----|-----|------|--------|
| Trigger (OR) | ... | ... | ... |
| Preselection | ... | ... | ... |
| ≥2 b-jets | ... | ... | ... |
| 0 leptons | ... | ... | ... |
| Δφ > 0.5 | ... | ... | ... |
| ΔMET(PF−calo) < 0.5 | ... | ... | ... |
| MET > 250 GeV | ... | ... | ... |

In [ ]:
# Example for a single sample:
# import pandas as pd
# df = pd.DataFrame({
#     "Cut": ["Trigger", "Presel", "≥2 b-jets", "0 leptons", "Δφ>0.5", "|ΔMET(PF−calo)|<0.5", "MET>250"],
#     "Yield": [N0, N1, N2, N3, N4, N5, N6]  # keys: trigger, presel, 0lep, Δφ>0.5, |ΔMET(PF−calo)|<0.5, MET>250
# })
# print(df)

### Exercise 3.3 — Yield table
Create a yield table for your sample(s). Discuss: which background survives most in the SR? Why?

In [ ]:
# Your code: yield table

---
## 7. Control Regions

- **Goal:** use **control regions** to validate background modelling with data.
- **Regions in this notebook:** **Z (dilepton)** and **Top (single-lepton)**, following the reference analysis.

**Preselection:**

- Trigger ($p_\mathrm{T}^\mathrm{miss}$ or single electron).

- Jets: $\geq 1$, leading $p_\mathrm{T} > 100$ GeV, others $p_\mathrm{T} > 30$ GeV, $|\eta| < 2.5$.

- Overlap removal, photon veto.

- $\min \Delta\phi(\mathrm{jet}, p_\mathrm{T}^\mathrm{miss}) > 0.5$

- $\Delta p_\mathrm{T}^\mathrm{miss}(\mathrm{PF} - \mathrm{calo}) < 0.5$

- $p_\mathrm{T}^\mathrm{miss}$ or Recoil $> 250$ GeV.
    - Recoil: $\mathbf{U} = -(\vec{p}_\mathrm{T}^\mathrm{miss} + \sum \vec{p}_{\mathrm{T}}^{\,\ell})$.





**Z CR (dilepton):**

- On top of preselection: exactly two opposite-sign same-flavor leptons (ee or $\mu\mu$).

- Leading lepton $p_\mathrm{T} > 30$ GeV; $70 < m_{\ell\ell} < 110$ GeV.

- No b-tag requirement.

- **Data can be plotted here.**

**Top CR (single-lepton):**

- On top of preselection: exactly one lepton (e or $\mu$) with $p_\mathrm{T} > 30$ GeV.

- $m_\mathrm{T} < 160$ GeV with $m_\mathrm{T} = \sqrt{2\,p_\mathrm{T}^\mathrm{miss}\,p_{\mathrm{T},\ell}\big(1-\cos\Delta\phi(p_\mathrm{T}^\mathrm{miss},\mathbf{p}_{\mathrm{T},\ell})\big)}$.

- Same b-tag as SR ($\geq 2$ b-jets); at least 2 additional jets that are not b-tagged.

- **Data can be plotted here.**


### Z control regions (dilepton split)

We split the Z CR into:
- **zecr**: exactly two electrons (OSSF), leading lepton $p_\mathrm{T}>30$ GeV, $70<m_{\ell\ell}<110$ GeV
- **zmucr**: exactly two muons (OSSF), leading lepton $p_\mathrm{T}>30$ GeV, $70<m_{\ell\ell}<110$ GeV

For **electron-based CR plots**, data is taken from `SingleElectron` datasets.

In [ ]:
# Z CR common definitions (recoil computed before final zecr/zmucr masks)
from processor.bbdm_processor import select_tight_electrons, select_tight_muons, recoil_pt, met_pf_calo_mask

PRESEL_RECOIL_MIN = 250.0
LEAD_JET_PT_MIN = 100.0
Z_CR_MLL_LO, Z_CR_MLL_HI = 70.0, 110.0
LEAD_LEP_PT_CR = 30.0
CR_SIGNAL_VIS_SCALE = 1.0e4
SIG_NAME = "bbDM_2HDMa_LO_5f"

def z_cr_components(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)
    nele, nmu = ak.count(tight_ele.pt, axis=1), ak.count(tight_mu.pt, axis=1)
    two_ee = (nele == 2) & (nmu == 0) & (ak.sum(tight_ele.charge, axis=1) == 0)
    two_mumu = (nele == 0) & (nmu == 2) & (ak.sum(tight_mu.charge, axis=1) == 0)

    tight_ele_pad = ak.pad_none(tight_ele, 2)
    tight_mu_pad = ak.pad_none(tight_mu, 2)
    pair_ee = tight_ele_pad[:, 0] + tight_ele_pad[:, 1]
    pair_mumu = tight_mu_pad[:, 0] + tight_mu_pad[:, 1]

    sum_lep_px = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.cos(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.cos(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    sum_lep_py = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.sin(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.sin(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))

    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)

    dphi_j = np.abs(good_jets.phi - met_phi)
    dphi_j = ak.where(dphi_j > np.pi, 2 * np.pi - dphi_j, dphi_j)
    min_dphi = ak.min(dphi_j, axis=1)
    dphi_cut = min_dphi > 0.5
    met_pf_calo_ok = met_pf_calo_mask(events, mode="cr", sum_lep_px=sum_lep_px, sum_lep_py=sum_lep_py)

    presel = (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN) & dphi_cut & met_pf_calo_ok & (recoil > PRESEL_RECOIL_MIN)
    mll_ee = ak.where(two_ee, ak.fill_none(pair_ee.mass, -1.0), ak.full_like(met_pt, -1.0))
    mll_mumu = ak.where(two_mumu, ak.fill_none(pair_mumu.mass, -1.0), ak.full_like(met_pt, -1.0))
    lead_lep_pt = ak.where(two_ee, ak.max(tight_ele.pt, axis=1), ak.where(two_mumu, ak.max(tight_mu.pt, axis=1), ak.full_like(met_pt, 0.0)))

    zecr = presel & two_ee & (lead_lep_pt > LEAD_LEP_PT_CR) & (mll_ee > Z_CR_MLL_LO) & (mll_ee < Z_CR_MLL_HI)
    zmucr = presel & two_mumu & (lead_lep_pt > LEAD_LEP_PT_CR) & (mll_mumu > Z_CR_MLL_LO) & (mll_mumu < Z_CR_MLL_HI)
    return zecr, zmucr, recoil, good_jets

In [ ]:
# zecr recoil: stacked backgrounds + signal overlays (SR style)
bins_recoil = np.array([250, 300, 400, 550, 1000], dtype=float)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    zecr, _, recoil, _ = z_cr_components(events_by_dataset[name])
    vals = ak.to_numpy(ak.ravel(recoil[zecr]))
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_recoil, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)

ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("Recoil [GeV]")
plt.ylabel("Events (scaled)")
plt.title("zecr (ee): Recoil")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# zmucr recoil: stacked backgrounds + signal overlays (SR style)
bins_recoil = np.array([250, 300, 400, 550, 1000], dtype=float)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    _, zmucr, recoil, _ = z_cr_components(events_by_dataset[name])
    vals = ak.to_numpy(ak.ravel(recoil[zmucr]))
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_recoil, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)

ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("Recoil [GeV]")
plt.ylabel("Events (scaled)")
plt.title("zmucr (mumu): Recoil")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# zecr dilepton mass (m_ll): stacked backgrounds (no signal overlay)
bins_mll = np.linspace(70, 110, 21)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    ev = events_by_dataset[name]
    zecr, _, _, _ = z_cr_components(ev)
    tight_ele = select_tight_electrons(ev)
    pair_ee = ak.pad_none(tight_ele, 2)[:, 0] + ak.pad_none(tight_ele, 2)[:, 1]
    vals = ak.to_numpy(ak.ravel(ak.fill_none(pair_ee.mass[zecr], -1.0)))
    vals = vals[vals > 0]
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_mll, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)
plt.xlabel("$m_{ee}$ [GeV]")
plt.ylabel("Events (scaled)")
plt.title("zecr (ee): dilepton mass")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# zmucr dilepton mass (m_ll): stacked backgrounds (no signal overlay)
bins_mll = np.linspace(70, 110, 21)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    ev = events_by_dataset[name]
    _, zmucr, _, _ = z_cr_components(ev)
    tight_mu = select_tight_muons(ev)
    pair_mumu = ak.pad_none(tight_mu, 2)[:, 0] + ak.pad_none(tight_mu, 2)[:, 1]
    vals = ak.to_numpy(ak.ravel(ak.fill_none(pair_mumu.mass[zmucr], -1.0)))
    vals = vals[vals > 0]
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_mll, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)
plt.xlabel("$m_{\\mu\\mu}$ [GeV]")
plt.ylabel("Events (scaled)")
plt.title("zmucr (mumu): dilepton mass")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# zecr cos(theta*): stacked backgrounds + signal overlays (SR style)
bins_cts = np.linspace(0, 1, 5)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    zecr, _, _, good_jets = z_cr_components(events_by_dataset[name])
    gj = good_jets[zecr]
    has_two = ak.num(gj) >= 2
    jp = ak.pad_none(gj, 2)
    j0, j1 = jp[:, 0], jp[:, 1]
    valid = has_two & ~ak.is_none(j1)
    if ak.sum(valid) == 0:
        continue
    vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_cts, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)

ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("$\\cos(\\theta)*$")
plt.ylabel("Events (scaled)")
plt.title("zecr (ee): cos θ*")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# zmucr cos(theta*): stacked backgrounds + signal overlays (SR style)
bins_cts = np.linspace(0, 1, 5)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    _, zmucr, _, good_jets = z_cr_components(events_by_dataset[name])
    gj = good_jets[zmucr]
    has_two = ak.num(gj) >= 2
    jp = ak.pad_none(gj, 2)
    j0, j1 = jp[:, 0], jp[:, 1]
    valid = has_two & ~ak.is_none(j1)
    if ak.sum(valid) == 0:
        continue
    vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_cts, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)

ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("$\\cos(\\theta)*$")
plt.ylabel("Events (scaled)")
plt.title("zmucr (mumu): cos θ*")
legend_stack()
plt.tight_layout()
plt.show()

### Top control regions (single-lepton split)

We split the Top CR into:
- **tecr**: exactly one electron, $p_\mathrm{T}>30$ GeV, $m_T<160$ GeV, $\geq2$ b-jets, $\geq2$ non-b jets
- **tmucr**: exactly one muon, same kinematic requirements

For **electron-based CR plots**, data is taken from `SingleElectron` datasets.

In [ ]:
from processor.bbdm_processor import met_pf_calo_mask

TOP_CR_MT_MAX = 160.0

def top_cr_components(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)

    one_ele = (ak.count(tight_ele.pt, axis=1) == 1) & (ak.count(tight_mu.pt, axis=1) == 0)
    one_mu = (ak.count(tight_ele.pt, axis=1) == 0) & (ak.count(tight_mu.pt, axis=1) == 1)

    lep_pt = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.pt), ak.where(one_mu, ak.firsts(tight_mu.pt), ak.full_like(met_pt, 0.0))), 0.0)
    lep_phi = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.phi), ak.where(one_mu, ak.firsts(tight_mu.phi), ak.full_like(met_pt, 0.0))), 0.0)
    sum_lep_px = lep_pt * np.cos(lep_phi)
    sum_lep_py = lep_pt * np.sin(lep_phi)

    # Recoil built before final tecr/tmucr masks.
    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)

    dphi_j = np.abs(good_jets.phi - met_phi)
    dphi_j = ak.where(dphi_j > np.pi, 2 * np.pi - dphi_j, dphi_j)
    min_dphi = ak.min(dphi_j, axis=1)
    dphi_cut = min_dphi > 0.5
    met_pf_calo_ok = met_pf_calo_mask(events, mode="cr", sum_lep_px=sum_lep_px, sum_lep_py=sum_lep_py)
    presel = (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN) & dphi_cut & met_pf_calo_ok & (recoil > PRESEL_RECOIL_MIN)

    dphi = np.abs(ak.to_numpy(met_phi) - ak.to_numpy(lep_phi))
    dphi = np.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    mt = ak.Array(np.sqrt(2.0 * ak.to_numpy(met_pt) * ak.to_numpy(lep_pt) * (1.0 - np.cos(dphi))))

    n_non_b = njets - nbjets
    common = presel & (lep_pt > LEAD_LEP_PT_CR) & (mt < TOP_CR_MT_MAX) & (nbjets >= 2) & (n_non_b >= 2)
    tecr = common & one_ele
    tmucr = common & one_mu
    return tecr, tmucr, recoil, good_jets

In [ ]:
# tecr recoil: stacked backgrounds + signal overlays (SR style)
bins_recoil = np.array([250, 300, 400, 550, 1000], dtype=float)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    tecr, _, recoil, _ = top_cr_components(events_by_dataset[name])
    vals = ak.to_numpy(ak.ravel(recoil[tecr]))
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_recoil, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)

ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("Recoil [GeV]")
plt.ylabel("Events (scaled)")
plt.title("tecr (electron): Recoil")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# tmucr recoil: stacked backgrounds + signal overlays (SR style)
bins_recoil = np.array([250, 300, 400, 550, 1000], dtype=float)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    _, tmucr, recoil, _ = top_cr_components(events_by_dataset[name])
    vals = ak.to_numpy(ak.ravel(recoil[tmucr]))
    if len(vals) == 0:
        continue
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_recoil, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)

ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("Recoil [GeV]")
plt.ylabel("Events (scaled)")
plt.title("tmucr (muon): Recoil")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# tecr cos(theta*): stacked backgrounds + signal overlays (SR style)
bins_cts = np.linspace(0, 1, 5)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    tecr, _, _, good_jets = top_cr_components(events_by_dataset[name])
    gj = good_jets[tecr]
    has_two = ak.num(gj) >= 2
    jp = ak.pad_none(gj, 2)
    j0, j1 = jp[:, 0], jp[:, 1]
    valid = has_two & ~ak.is_none(j1)
    if ak.sum(valid) == 0:
        continue
    vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_cts, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)

ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("$\\cos(\\theta)*$")
plt.ylabel("Events (scaled)")
plt.title("tecr (electron): cos θ*")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# tmucr cos(theta*): stacked backgrounds + signal overlays (SR style)
bins_cts = np.linspace(0, 1, 5)
arrs, warrs, labs, stack_keys = [], [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    _, tmucr, _, good_jets = top_cr_components(events_by_dataset[name])
    gj = good_jets[tmucr]
    has_two = ak.num(gj) >= 2
    jp = ak.pad_none(gj, 2)
    j0, j1 = jp[:, 0], jp[:, 1]
    valid = has_two & ~ak.is_none(j1)
    if ak.sum(valid) == 0:
        continue
    vals = ak.to_numpy(ak.ravel(cos_theta_star(j0[valid], j1[valid])))
    arrs.append(vals)
    warrs.append(np.full_like(vals, norm, dtype=float))
    labs.append(latex_labels.get(name, labels.get(name, name)))
    stack_keys.append(name)

plt.figure(figsize=(6, 4))
if arrs:
    plt.hist(arrs, bins=bins_cts, weights=warrs, label=labs, color=[PROCESS_COLORS.get(k, DEFAULT_STACK_COLOR) for k in stack_keys], stacked=True, histtype="stepfilled", alpha=0.9)

ax = plt.gca()
ax.set_yscale("log")
ymin, ymax = ax.get_ylim()
ax.set_ylim(max(1e-3, ymin), ymax * 100.0)
plt.xlabel("$\\cos(\\theta)*$")
plt.ylabel("Events (scaled)")
plt.title("tmucr (muon): cos θ*")
legend_stack()
plt.tight_layout()
plt.show()

In [ ]:
# Region-wise cutflow / yields after trigger (SR, zecr, zmucr, tecr, tmucr)
print("Region yields by dataset:")
for name, ev0 in events_by_dataset.items():
    hlt_fields = set(ev0.HLT.fields) if hasattr(ev0, "HLT") and hasattr(ev0.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev0.event, dtype=bool)
    for tname in get_trigger_list():
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev0.HLT[tname]
    ev = ev0[trigger_mask]

    # SR final
    sr_counts = run_cutflow(ev)
    recoil_key = f"Recoil>{int(RECOIL_MIN)}"

    # CR finals
    zecr, zmucr, _, _ = z_cr_components(ev)
    tecr, tmucr, _, _ = top_cr_components(ev)

    print(labels.get(name, name), {
        "trigger": int(len(ev)),
        "sr_final": int(sr_counts[recoil_key]),
        "zecr": int(ak.sum(zecr)),
        "zmucr": int(ak.sum(zmucr)),
        "tecr": int(ak.sum(tecr)),
        "tmucr": int(ak.sum(tmucr)),
    })

# Extra visibility for signal mass points in each region
if SIG_NAME in events_by_dataset:
    ev0 = events_by_dataset[SIG_NAME]
    hlt_fields = set(ev0.HLT.fields) if hasattr(ev0, "HLT") and hasattr(ev0.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev0.event, dtype=bool)
    for tname in get_trigger_list():
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev0.HLT[tname]
    ev = ev0[trigger_mask]
    zecr, zmucr, recoil_z, gj_z = z_cr_components(ev)
    tecr, tmucr, recoil_t, gj_t = top_cr_components(ev)
    print("\nSignal masspoint yields:")
    for label, br in SIG_BRANCHES:
        if br not in ev.fields:
            continue
        m = ev[br] > 0
        print(label, {
            "sr": int(run_cutflow(ev[m])[f"Recoil>{int(RECOIL_MIN)}"]),
            "zecr": int(ak.sum(zecr & m)),
            "zmucr": int(ak.sum(zmucr & m)),
            "tecr": int(ak.sum(tecr & m)),
            "tmucr": int(ak.sum(tmucr & m)),
        })

### Discussion
What physics process produces large MET in Z→νν? Why is Z→νν a major background in our signal region?

---
## 8. Using the Coffea Processor

The provided **bbdm_processor.py** does the full selection and fills histograms. You can run it on one or more files to get merged results.

Example (pseudo-code):
```python
from processor.bbdm_processor import bbDMProcessor
from coffea.processor import run_uproot_job
files = {"ttbar": ["file1.root"]}
out = run_uproot_job(files, "events", bbDMProcessor(), executor=iterative_executor)
```
Then inspect `out["recoil"]`, `out["cutflow"]`, etc.

In [ ]:
# Optional: run the processor (uncomment and set paths)
# import sys
# sys.path.insert(0, "processor")
# from bbdm_processor import bbDMProcessor, get_nanoevents
# proc = bbDMProcessor()
# events = get_nanoevents("path/to/file.root")
# result = proc.process(events)
# print("Cutflow:", result["cutflow"])
# result["recoil"].plot()

---
## 9. Comparison Plots

Produce **comparison plots**: e.g. MET distribution **before** any SR cuts (but after ≥1 jet) and **after** full SR selection. This shows how the cut shapes the distribution.

In [ ]:
# Example: MET before and after SR
# met_presel = met[njets >= 1]
# recoil = met[sr_mask]
# fig, ax = plt.subplots(1, 2, figsize=(10, 4))
# ax[0].hist(ak.to_numpy(met_presel), bins=50, range=(0, 500), label="Presel", alpha=0.7)
# ax[1].hist(ak.to_numpy(recoil), bins=50, range=(200, 600), label="SR", alpha=0.7)
# ax[0].set_xlabel("MET [GeV]"); ax[1].set_xlabel("MET [GeV]")
# legend_stack(); plt.tight_layout(); plt.show()

### Exercise 3.4 — Comparison plots
Make a comparison: MET after preselection (≥1 jet) vs MET in signal region. Use two side-by-side or overlaid histograms with legends.

In [ ]:
# Your code: comparison plot

---
## 10. Summary — Session 3

- **Signal region:** Δφ(jet,MET) and PF–calo MET consistency, then **MET > 250 GeV**, ≥2 b-jets (leading two), 0 leptons.
- **Cutflow** and **yield tables** show how each cut reduces events; Z→νν and tt̄ dominate in the SR.
- **Control regions** (e.g. single-lepton) validate background modelling.
- The **bbdm_processor** encapsulates the full selection for reuse.

**End of the long exercise.** You have implemented a simplified bbDM search from data loading to signal region and yields. In a real analysis, next steps would include systematic uncertainties, statistical interpretation, and possibly limit setting.

### Optional: Vary the MET threshold
Re-run your cutflow and MET plot with MET > 150 GeV and MET > 250 GeV. How do the yields and shapes change? Why is the MET cut important for background rejection?

## Plot All Variables from the Full Output

Run the cell below to render all variables from the normalized full-analysis pickle:


In [ ]:
import importlib
import config.plot_pkl_variables as pp
pp = importlib.reload(pp)
pp.plot_all_variables_grid("output/output_2017_full.pkl")
